# 02 · Référentiel de blocs et enrichissement sémantique (P2)

**Problème.** Le sujet suggère la structure `BookID | Genre | Title | Author | Synopsis | Themes | Style | Keywords`. Le dataset Book Depository n'a **ni thèmes, ni style, ni mots-clés**, et ses ~1 000 catégories sont bruitées. On les **dérive par analyse sémantique** (embeddings SBERT) au lieu de les saisir à la main.

**Plan :** 1) taxonomie `blocks.json` → 2) catégories → genres → 3) zero-shot thèmes/style/atmosphère → 4) mots-clés → 5) validation → 6) export.

In [ ]:
# --- Imports et chemins (le notebook s'exécute depuis notebooks/, on remonte à la racine du repo)
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import enrichment as en

FIG = ROOT / "docs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/books_clean.parquet")   # catalogue de P1 (12 000 livres)
print(df.shape)

## 1. La taxonomie (`blocks.json`)
4 blocs (Genre, Thèmes, Style, Atmosphère), chacun avec des labels **et une phrase descriptive par label**. C'est la *phrase* qu'on encode : « boy wizard school » est ainsi proche de *Fantasy* sans que le mot apparaisse. La taxonomie s'inspire des grandes familles de genres de l'édition (BISAC / rayons de librairie) et de thèmes classiques de l'analyse littéraire. Les 6 labels d'**époque** (Ancient world … Future) alimentent aussi la question Q7 du questionnaire.

In [ ]:
blocks = en.load_blocks()                       # valide aussi que chaque label a une vraie phrase
pd.DataFrame({b: [len(v)] for b, v in blocks.items()}, index=["nb de labels"])

In [ ]:
# Exemple : un label et sa phrase descriptive (c'est elle qui est encodée)
for b in en.BLOCK_NAMES:
    l = next(iter(blocks[b])); print(f"{b:11s} | {l:25s} | {blocks[b][l]}")

## 2. Embeddings (modèle `all-MiniLM-L6-v2`, comme P3)
Vecteurs **L2-normalisés** : le produit scalaire est alors égal au cosinus. Les embeddings sont mis en cache sur disque (relancer est instantané). Quand `EmbeddingEngine` (P3) sera prêt, il suffit de le passer à la place de `SbertEncoder` (même méthode `encode`).

In [ ]:
encoder = en.SbertEncoder()                                                   # chargement paresseux
book_emb = en.encode_cached(encoder, df["embed_text"].fillna("").tolist(),
                            df["book_id"].astype(str).tolist(), "p2_all-MiniLM-L6-v2_clean_books")
enc_blocks = en.encode_blocks(encoder, blocks)
print(book_emb.shape, {b: m.shape for b, (_, m) in enc_blocks.items()})

## 3. Catégories Book Depository → Genres
Les catégories sont nombreuses et bruitées (« Fiction », « Teaching Resources »…). Stratégie en 2 temps : des **règles mots-clés** pour les cas évidents (« Fantasy Books » → Fantasy) puis, pour le reste, le **genre le plus proche par embeddings** (accepté seulement si cosinus ≥ 0,30, sinon la catégorie est jugée trop générique). La table est exportée dans `data/reference/category_genre_map.csv` avec une colonne **`genre_manual`** : on y relit à la main les 100 catégories les plus fréquentes ; elle est conservée à chaque régénération.

In [ ]:
counts = en.category_counts(df)
cat_map = en.build_category_map(counts, encoder, blocks)
cat_map.to_csv(en.CATEGORY_MAP_PATH, index=False)
cat_map.head(30)[["category", "n_books", "rule_genres", "emb_genre", "emb_sim", "genre_auto"]]

In [ ]:
# Part des catégories (et des occurrences) résolues par règle / par embeddings / non résolues
has_rule = cat_map["rule_genres"] != ""
has_auto = cat_map["genre_auto"] != ""
print(f"règles: {has_rule.mean():.0%} | embeddings seuls: {(~has_rule & has_auto).mean():.0%} | non résolues: {(~has_auto).mean():.0%} des catégories")
w = cat_map["n_books"]
print(f"pondéré par les occurrences : résolues {w[has_auto].sum() / w.sum():.0%}")

> **À faire (manuel, ~20 min)** : ouvrir `category_genre_map.csv`, relire les 100 premières lignes, corriger via `genre_manual` (labels séparés par `|`), puis relancer la cellule précédente / le script : la relecture est conservée.

In [ ]:
genre_labels, genre_emb = enc_blocks["Genre"]
genres, g_stats = en.assign_genres(df, book_emb, genre_labels, genre_emb, cat_map)
print(g_stats)
df.assign(genres=genres)[["title", "categories", "genres"]].sample(8, random_state=1)

## 4. Zero-shot par embeddings : Thèmes, Style, Atmosphère
Pour chaque livre : cosinus entre l'embedding de sa description et celui de la phrase de chaque label. Deux précautions :

* **z-score par label** : certaines phrases sont proches de *tous* les livres. On compare donc chaque livre au livre moyen pour ce label.
* **Seuil calibré** (`z_min`) et au moins 1 label garanti. Atmosphère = **1 époque + 1 ambiance**.

In [ ]:
# Calibration du seuil : nb moyen de labels gardés selon z_min (on vise ~ top_k pour Thèmes, sans labels « forcés » partout)
for block in ["Themes", "Style", "Atmosphere"]:
    labels, emb = enc_blocks[block]
    print(block); display(en.threshold_report(en.score_block(book_emb, emb), labels, block))

In [ ]:
assigned, sims_by_block = {}, {"Genre": (genre_labels, en.score_block(book_emb, genre_emb))}
for block in ["Themes", "Style", "Atmosphere"]:
    labels, emb = enc_blocks[block]
    assigned[block], sims = en.assign_labels(book_emb, labels, emb, block, z_min=en.Z_MIN)
    sims_by_block[block] = (labels, sims)
df.assign(themes=assigned["Themes"], styles=assigned["Style"], atmospheres=assigned["Atmosphere"])[
    ["title", "themes", "styles", "atmospheres"]].sample(8, random_state=1)

**Optionnel (discussion qualité/temps)** : comparer sur ~200 livres avec un vrai zero-shot NLI (`facebook/bart-large-mnli`). Il est nettement plus lent (une passe du modèle par couple livre × label) ; nos embeddings donnent un classement de tout le catalogue en quelques secondes.

In [ ]:
# (Optionnel) zero-shot NLI sur 200 livres : plusieurs minutes sur CPU, à lancer si on a le temps.
RUN_NLI = False
if RUN_NLI:
    import time
    from transformers import pipeline
    nli = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    sub = df.sample(200, random_state=0)
    theme_labels = list(blocks["Themes"])
    t0 = time.time()
    nli_top = [nli(t[:600], theme_labels, multi_label=True)["labels"][:3] for t in sub["description"]]
    print(f"NLI: {time.time() - t0:.0f}s pour 200 livres")

## 5. Mots-clés (principe KeyBERT)
On garde l'algorithme de KeyBERT (cosinus candidat↔document + **MMR**, diversité 0,5, 1-2 mots) mais on **pré-filtre les candidats par TF-IDF** : KeyBERT standard encoderait tout le vocabulaire de la collection (centaines de milliers de phrases), trop lent sur CPU. Résultat identique en principe, en quelques minutes.

In [ ]:
keywords = en.extract_keywords(df, book_emb, encoder)
df.assign(keywords=keywords)[["title", "keywords"]].sample(8, random_state=1)

In [ ]:
# (Optionnel) comparaison avec la vraie bibliothèque KeyBERT sur 5 livres
sub = df.sample(5, random_state=2)
for t, mine in zip(sub["title"], [keywords[i] for i in sub.index]):
    print(t[:40], "| à nous:", mine)
# lib = [en.extract_keywords_keybert_lib([d]) for d in sub["description"]]   # décommenter si keybert est installé

## 6. Assemblage et export (contrat `books_enriched.parquet`)

In [ ]:
enriched, report = en.enrich_catalogue(df, encoder, blocks, cat_map, book_emb)
en.save_enriched(enriched, ROOT / "data/processed/books_enriched.parquet")
print(report)

## 7. Visualisations (rapport)
Répartition des genres et des thèmes, puis **heatmap genre × thème** : elle illustre les « relations entre catégories » demandées par le sujet (ex. *Mystery* ↔ *Crime & justice*).

In [ ]:
from collections import Counter
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, (col, title) in zip(ax, [("genres", "Genres"), ("themes", "Thèmes")]):
    c = Counter(l for ls in enriched[col] for l in ls); s = pd.Series(c).sort_values()
    s.plot.barh(ax=a); a.set_title(f"{title} : nb de livres"); 
plt.tight_layout(); plt.savefig(FIG / "02_genres_themes_distribution.png", dpi=150); plt.show()

In [ ]:
# Co-occurrence genre x thème (nb de livres portant les deux labels)
G, T = list(blocks["Genre"]), list(blocks["Themes"])
M = pd.DataFrame(0, index=G, columns=T)
for gs, ts in zip(enriched["genres"], enriched["themes"]):
    for g in gs:
        for t in ts: M.loc[g, t] += 1
Mn = M.div(M.sum(1).replace(0, 1), axis=0)             # normalisé par genre : « profil thématique » de chaque genre
fig, ax = plt.subplots(figsize=(13, 8)); im = ax.imshow(Mn.values, cmap="viridis")
ax.set_xticks(range(len(T)), T, rotation=75, ha="right"); ax.set_yticks(range(len(G)), G)
ax.set_title("Genre × Thème (part des livres du genre portant le thème)"); plt.colorbar(im)
plt.tight_layout(); plt.savefig(FIG / "02_heatmap_genre_theme.png", dpi=150); plt.show()

## 8. Validation sur 60 livres annotés à la main
1. `python -m scripts.run_enrichment --annotation-sheet` écrit `data/reference/gold_60_to_annotate.csv` (**sans** les prédictions, pour ne pas biaiser).
2. P2 et P6 remplissent `gold_genres` et `gold_themes` (labels de `blocks.json`, séparés par `|`) et enregistrent sous `data/reference/gold_60.csv`.
3. On mesure precision@k, recall, « le 1er label est dans l'annotation » et « au moins un label correct ».

In [ ]:
gold_path = ROOT / "data/reference/gold_60.csv"
if gold_path.exists():
    res = en.evaluate_against_gold(enriched, gold_path); display(pd.DataFrame(res).T)
else:
    print("gold_60.csv absent : voir la procédure ci-dessus.")

In [ ]:
# 10 bons exemples + 5 erreurs : à choisir et commenter à la main pour le rapport
view = enriched[["title", "description", "genres", "themes", "styles", "atmospheres", "keywords"]].sample(15, random_state=7)
view.assign(description=view["description"].str.slice(0, 160))

## Conclusion
Le catalogue est enrichi de `genres`, `themes`, `styles`, `atmospheres`, `keywords` et `label_scores` **sans annotation manuelle à grande échelle** : tout vient de la sémantique des descriptions. Limites : les descriptions sont des textes marketing (biais de ton), le z-score suppose une distribution raisonnablement unimodale, et la tronquature à 256 tokens de MiniLM ignore la fin des longues descriptions. Pistes : MPNet, zero-shot NLI sur les cas ambigus, correction manuelle ciblée.